# Exp125 v2 - can we train a second fold in a Kaggle kernel?

Diagnostic. **No submission, no slot.**

Answers three things with measurements rather than estimates:

1. **Is the incumbent checkpoint 50 epochs or 400?** The pack is named
   `...-50ep-v1` but its own `ARTIFACT_MANIFEST.json` says
   `artifact_name: biohub-tracking-support-pack-400ep-snapshot-v1`, and our TTA
   patch log prints "400ep". This matters: "train longer" is only an obvious win
   if the incumbent really is 50.
2. **How long does training actually take** on a T4, separating one-off startup
   from per-iteration cost.
3. **Does a second fold fit** in the 9h kernel limit, and what iteration budget
   should it use.

## Why a second fold is the goal

`edge_predictor_best.pth` contains BOTH learned stages - the `TemporalUNet3D`
detector and the `SimpleNodeTransformer` linker. Everything we have tuned so far
(ILP costs, motion relink, gap closing, divisions) sits downstream of it. A
second independently-trained model would let us ensemble at the probability-map
level, which improves detection - the half we have never touched.

## Two fixes over v1

- v1 died with `OSError: Read-only file system`. `dataspec.py` sets
  `WEIGHTS_PATH = Path(__file__).parent.parent / "weights"`, i.e. relative to the
  script, which lives in a read-only `/kaggle/input` dataset. Fixed by copying
  the repo to `/kaggle/working` first.
- v1 amortised startup into per-iteration cost. This version times two runs of
  different length and solves for both terms separately.


In [ ]:
try:
    import zarr, geff, tracksdata
    
except: 
    import os
    import sys
    import subprocess
    import importlib
    import importlib.util
    from pathlib import Path
    
    # Polars wheel in the support package uses the 32-bit runtime package.
    os.environ.setdefault("POLARS_PREFER_PKG", "32")
    
    SUPPORT_DIR = Path(
        "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1"
    )
    WHEELS_DIR = SUPPORT_DIR / "wheels"
    
    if not WHEELS_DIR.exists():
        # Fallback for Kaggle's shorter mounted dataset path.
        candidates = list(Path("/kaggle/input").glob("**/wheels"))
        if not candidates:
            raise FileNotFoundError("Could not find the attached offline wheels directory")
        WHEELS_DIR = candidates[0]
    
    print("Offline wheels:", WHEELS_DIR)
    
    
    # These are packages supplied by the attached dataset.
    # Deliberately exclude:
    #   numpy, scipy, torch, pandas, dask, xarray, numba, llvmlite
    #
    # Kaggle already supplies compatible versions of those packages.
    OFFLINE_PACKAGES = [
        "tracksdata",
        "zarr==3.2.1",
        "numcodecs==0.15.1",
        "donfig==0.8.1.post1",
        "geff==1.2.0.1.1",
        "geff-spec==1.1.1",
        "pyscipopt==6.2.1",
        "ilpy==0.6.0",
        "rustworkx==0.18.0",
        "polars==1.42.0",
        "polars-runtime-32==1.42.0",
        "bidict==0.23.1",
        "imagecodecs==2026.6.26",
    ]
    
    
    def module_missing(module_name: str) -> bool:
        return importlib.util.find_spec(module_name) is None
    
    
    # Import name differs from pip package name in several cases.
    REQUIRED_IMPORTS = {
        "tracksdata": "tracksdata",
        "zarr": "zarr",
        "numcodecs": "numcodecs",
        "geff": "geff",
        "pyscipopt": "pyscipopt",
        "ilpy": "ilpy",
        "rustworkx": "rustworkx",
        "polars": "polars",
        "imagecodecs": "imagecodecs",
    }
    
    
    def purge_modules(module_roots):
        """
        Remove already-imported package modules from sys.modules.
    
        Normally this cell runs before imports, but this also protects against
        accidental imports performed by earlier Kaggle initialization code.
        """
        for root in module_roots:
            for name in list(sys.modules):
                if name == root or name.startswith(root + "."):
                    sys.modules.pop(name, None)
    
    
    def install_offline_packages():
        cmd = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-index",
            "--no-deps",
            "--find-links",
            str(WHEELS_DIR),
            *OFFLINE_PACKAGES,
        ]
    
        print("Installing attached packages without modifying NumPy/SciPy/Torch...")
        result = subprocess.run(
            cmd,
            text=True,
            capture_output=True,
        )
    
        if result.returncode != 0:
            print(result.stdout[-4000:])
            print(result.stderr[-4000:])
            raise RuntimeError("Offline dependency installation failed")
    
        purge_modules(REQUIRED_IMPORTS.values())
    
    
    install_offline_packages()
    
    
    # Verify the complete import chain needed by biohub_tracking.io.
    failures = {}
    
    for name, module_name in {
        **REQUIRED_IMPORTS,
        "numpy": "numpy",
        "scipy": "scipy",
        "dask": "dask.array",
        "xarray": "xarray",
    }.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    
    if failures:
        raise ImportError(
            "Dependency verification failed:\n"
            + "\n".join(f"{name}: {error}" for name, error in failures.items())
        )

import zarr, geff, tracksdata
print("All offline dependencies imported successfully.")

In [ ]:
# Locate repo + data, and copy the repo somewhere WRITABLE.
import os, sys, glob, json, time, shutil, subprocess
from pathlib import Path

_m=[]
for pat in ("/kaggle/input/*/*/*/src/tracking_cellmot/metrics.py",
            "/kaggle/input/*/*/src/tracking_cellmot/metrics.py",
            "/kaggle/input/**/tracking_cellmot/metrics.py"):
    _m+=glob.glob(pat,recursive=True)
assert _m, "official scorer dataset not attached"
SRC_RO=Path(_m[0]).parents[1]; REPO_RO=SRC_RO.parent

REPO=Path("/kaggle/working/repo")
if REPO.exists(): shutil.rmtree(REPO)
shutil.copytree(REPO_RO, REPO)
SRC=REPO/"src"
(REPO/"weights").mkdir(exist_ok=True)          # WEIGHTS_PATH now writable
print("writable repo:", REPO)

COMP=Path("/kaggle/input/competitions/biohub-cell-tracking-during-development")
if not COMP.exists(): COMP=Path(glob.glob("/kaggle/input/*biohub-cell-tracking*")[0])
TRAIN=COMP/"train"
stems=sorted(p.name[:-5] for p in TRAIN.glob("*.zarr") if (TRAIN/f"{p.name[:-5]}.geff").exists())
print(f"labelled movies: {len(stems)}")
print("GPU:", os.popen("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader").read().strip())


## 1. How many epochs is the incumbent checkpoint?

`checkpoint_last.pth` is 25 MB against `edge_predictor_best.pth`'s 8.4 MB - the
extra bulk is usually optimiser state, which often carries an epoch counter.

In [ ]:
import torch
w = glob.glob("/kaggle/input/**/edge_predictor_best.pth", recursive=True)
l = glob.glob("/kaggle/input/**/checkpoint_last.pth", recursive=True)
for label, paths in (("edge_predictor_best", w), ("checkpoint_last", l)):
    if not paths:
        print(f"{label}: not found"); continue
    d = torch.load(paths[0], map_location="cpu", weights_only=False)
    print(f"--- {label} ({os.path.getsize(paths[0])/1e6:.1f} MB) ---")
    if isinstance(d, dict):
        for k, v in d.items():
            if hasattr(v, "shape"):        print(f"    {k}: tensor{tuple(v.shape)}")
            elif isinstance(v, dict):      print(f"    {k}: dict[{len(v)}] e.g. {list(v)[:4]}")
            elif isinstance(v, (int,float,str)): print(f"    {k}: {v}   <-- scalar")
            else:                          print(f"    {k}: {type(v).__name__}")
    else:
        print("    plain state_dict, no metadata -> epoch count NOT recoverable")


## 2. Time two runs of different length

`train_one_epoch` cycles the loader when `max_iters` is set, so an "epoch" is
exactly `max_iters` batches. Timing `N1` and `N2` iterations lets us solve

    t = startup + per_iter * N

for both terms instead of conflating them.

In [ ]:
BATCH = int(os.environ.get("CAL_BATCH", "4"))
N1, N2 = 6, 24

def run(n_iters):
    env = dict(os.environ); env["PYTHONPATH"] = str(SRC)
    cmd = [sys.executable, str(REPO/"scripts"/"train_unet_transformer.py"),
           "--data-dir", str(TRAIN), "--split", "0", "--epochs", "1",
           "--max-iters", str(n_iters), "--batch-size", str(BATCH),
           "--num-workers", "2", "--single-gpu"]
    t0 = time.time()
    p = subprocess.run(cmd, cwd=str(REPO), env=env, capture_output=True, text=True)
    dt = time.time() - t0
    print(f"  max_iters={n_iters:>3}  rc={p.returncode}  {dt:.1f}s")
    if p.returncode != 0:
        print(p.stdout[-1500:]); print(p.stderr[-2500:])
    return dt, p

print("timing runs (batch size %d):" % BATCH)
t1, p1 = run(N1)
t2, p2 = run(N2)
OK = (p1.returncode == 0 and p2.returncode == 0)
print("\n--- stdout of the longer run ---")
print(p2.stdout[-2500:])


In [ ]:
# 3. Projection
if not OK:
    print("Runs FAILED - see stderr above. No projection.")
else:
    per_iter = (t2 - t1) / (N2 - N1)
    startup  = t1 - per_iter * N1
    print(f"per-iteration : {per_iter:6.2f} s   (batch {BATCH})")
    print(f"startup       : {startup:6.1f} s   (imports, dataset indexing, model init)")
    LIMIT_H = 9
    budget_s = LIMIT_H*3600 - startup - 600      # 10 min headroom for saving/eval
    total_iters = int(budget_s / per_iter)
    print(f"\nIn a {LIMIT_H}h kernel, after startup and 10 min headroom:")
    print(f"  total trainable iterations : {total_iters:,}")
    print(f"  images seen (batch {BATCH})     : {total_iters*BATCH:,}")
    print()
    print("  epochs achievable at a given --max-iters budget:")
    for ipe in (100, 250, 500, 1000):
        print(f"     --max-iters {ipe:>5} -> {total_iters//ipe:>5} epochs")
    print()
    print("  NOTE: with --max-iters set, an 'epoch' is exactly that many batches,")
    print("        so epoch count is not comparable to a full-dataset epoch.")


## 4. A second fold needs a custom splits file

With no splits file the script builds a **single** deterministic seed-0 fold, so
`--split 1` would raise `IndexError`. Writing our own splits JSON is the whole
fix - and a different partition is what makes the second model genuinely
independent rather than just a different random seed.

In [ ]:
import random
SPLITS = Path("/kaggle/working/dataset_splits.json")
folds = []
for fold in range(3):
    s = list(stems); random.Random(100 + fold).shuffle(s)
    n_val = max(1, len(s)//10)
    folds.append({"split": fold, "train": s[n_val:], "test": s[:n_val]})
SPLITS.write_text(json.dumps(folds, indent=1))
print(f"wrote {SPLITS} with {len(folds)} folds")
for f in folds:
    print(f"  fold {f['split']}: {len(f['train'])} train / {len(f['test'])} val")
ov = set(folds[0]["test"]) & set(folds[1]["test"])
print(f"\nfold0/fold1 validation overlap: {len(ov)} movies "
      f"({len(ov)/len(folds[0]['test']):.0%}) - lower is more independent")
print("\nTo train it:  --splits /kaggle/working/dataset_splits.json --split 1")


In [ ]:
for _p in ["submission.csv", "/kaggle/working/submission.csv"]:
    assert not os.path.exists(_p), f"unexpected submission at {_p}"
print("confirmed: no submission.csv; costs no submission slot")
